# 02. Learning one length with REINFORCE

The only trainable part is the table $Q$ (answer tag $\times$ prompt tag), starting at zero. At slot $k$ the attention is $\mathrm{softmax}(Q[a_k, \cdot])$ over the instructions. Reward is 1 if the final state is right. The update is REINFORCE with a running baseline $b$ and the paper's length normalisation:

$$Q \leftarrow Q + \eta\, \nabla_Q\, \frac{1}{B}\sum_i (r_i - b)\,\frac{\log \pi(\text{chain}_i)}{L_i}.$$

In [ ]:
import sys; sys.path.append('..')   # the code lives in the repo root
import matplotlib.pyplot as plt
from model import load_box
from train import train, takeoff
from plots import plot_history, plot_Q
box = load_box()

## Length 5, three seeds

Watch for a long plateau near chance (1/96) and then a sudden jump: the grokking-like transition from the paper.

In [ ]:
runs = [train(box, [5], steps=600, seed=s, verbose=False) for s in range(3)]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for s, (Q, h) in enumerate(runs):
    axes[0].plot(h['step'], h['success'][5], lw=2, label=f'seed {s}')
    axes[1].plot(h['step'], h['hit'][5], lw=2, label=f'seed {s}')
axes[0].set_title('success at L = 5', loc='left'); axes[1].set_title('attention on the right instruction', loc='left')
for ax in axes: ax.set_xlabel('training step'); ax.grid(alpha=0.3); ax.legend()
plt.show()
print('take-off step (success >= 0.5):', [takeoff(h, 5) for _, h in runs])

## What Q learned

Reorder the columns so the true pairs sit on the diagonal. A bright diagonal means the codebook has been found.

In [ ]:
from data import make_codebook
Q, h = runs[0]
plot_Q(Q, make_codebook(seed=0)); plt.show()

## A long length on its own

The paper predicts that training on a long horizon alone stalls: success is near zero, so there is almost no signal. Same code, $L=15$.

In [ ]:
Q15, h15 = train(box, [15], steps=400, verbose=False)
plot_history(h15, 'L = 15 trained alone'); plt.show()